In [ ]:
%matplotlib inline

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt


# Bulgarian Electricity Prices: A Statistical Analysis (2015–2025)

## 1. Introduction

### 1.1 Real-World Significance

Electricity pricing is one of the most consequential economic signals in modern societies. Day-ahead prices — set on wholesale markets one day in advance — determine the revenue of power generators, the costs of industrial consumers, and increasingly, the bills of households on dynamic tariffs. Understanding what drives these prices is relevant to energy policy, market regulation, and economic planning.

Bulgaria presents a particularly interesting case. As a net electricity exporter in the region, Bulgaria's wholesale market is influenced by a combination of domestic generation mix (nuclear, hydro, thermal, and growing renewables), cross-border flows with neighboring markets, and weather-driven demand fluctuations. The period 2015–2025 encompasses extraordinary market conditions: stable pre-crisis years, the COVID-19 demand shock of 2020, the extreme price volatility of 2021–2022 driven by the European gas crisis and Russia's invasion of Ukraine, and the subsequent post-crisis normalization.

### 1.2 Literature Review

The relationship between weather variables and electricity prices is well-established in energy economics research. Boogen et al. (2024) estimate the effect of temperature, wind speed, solar radiation, and precipitation on wholesale electricity prices across six European countries, finding that weather impacts prices in a nonlinear manner and identifying thresholds at which weather effects amplify significantly. Haluška et al. (2025) analyze meteorological variables including temperature, wind speed, and humidity on electricity prices across Central European countries using ENTSO-E data from 2019 to 2024, finding that results are country-specific and asymmetric. Trebbien et al. (2024) provide a comprehensive analysis of patterns and correlations in European day-ahead electricity prices between 2019 and 2023, noting the period was characteristically abnormal due to the energy crisis and that weather is a common driving factor across bidding zones.

No study to our knowledge focuses specifically on Bulgaria over a decade-long period that includes the 2021–2022 energy crisis. This analysis aims to fill that gap.

### 1.3 Research Questions

This analysis addresses three research questions:

1. *How strongly do weather variables correlate with real (inflation-adjusted) Bulgarian day-ahead electricity prices?*
2. *Which weather variable has the strongest independent relationship with price?*
3. *Did the 2021–2022 energy crisis fundamentally alter the weather-price relationship, and if so, how?*

### 1.4 Scope and Limitations

This analysis identifies **statistical associations** only. No causal claims are made. The observed correlations are consistent with known market mechanisms — weather affects demand, which affects price — but this study does not establish causation. The data is observational, not experimental, and multiple confounding factors operate simultaneously in electricity markets.

Additionally, day-ahead electricity prices are determined by the merit order mechanism, in which generators bid into the market from cheapest to most expensive and the marginal generator sets the price. While this mechanism is known in principle, the actual bids, marginal costs, and fuel prices on any given day are not publicly available. This analysis therefore measures the **net observable effect** of weather on price through all channels simultaneously.

## 2. Data Sources

This analysis draws on four independent data sources, combined to form a single daily time series spanning 2015–2025. The first three answer the substantive questions of the analysis: *what did electricity cost on the wholesale market?*, *what was the weather?*, and *what was a euro actually worth?* The fourth — city population data — supports the construction of a national weather index from city-level weather observations. No single source could answer all of these questions, and substituting any one of them with a derived or modeled equivalent would compromise the analysis.

### 2.1 Source 1 — Ember: Day-Ahead Electricity Prices

Ember is an independent climate and energy think tank that publishes a curated dataset of European wholesale day-ahead electricity prices, sourced originally from ENTSO-E's Transparency Platform. The dataset provides daily clearing prices in nominal euros per megawatt-hour (€/MWh) for Bulgaria, covering the full period 2015 through 2025.

The day-ahead market is the central price-discovery mechanism for *wholesale* electricity in Bulgaria. Each day, generators and large buyers submit bids for the following day's hourly delivery, and the market operator (IBEX, the Bulgarian Independent Energy Exchange) clears them into a single price per hour. Ember aggregates these hourly clearing prices into a daily mean, which is the variable used throughout this notebook. Daily resolution is appropriate here because the weather data will also be aggregated to daily resolution, and because the research questions concern multi-year patterns rather than intraday dynamics.

It is worth being explicit about what this price represents and what it does not. The day-ahead price is the wholesale market clearing price — what generators receive for selling electricity to large buyers (utilities, industrial consumers, traders) one day in advance. It reflects the marginal cost of the last unit of electricity needed to meet expected demand, typically a gas-fired plant during peak hours. It does not reflect what Bulgarian households pay on their electricity bills. Household prices in Bulgaria are largely regulated by KEVR (the Energy and Water Regulatory Commission), updated only a few times per year, and include network charges, taxes, and supplier margins on top of the underlying energy cost. Long-term bilateral contracts negotiated outside the exchange are also not reflected in this dataset.

This distinction matters for interpreting the results. The analysis will reveal how the *wholesale market* prices weather risk — how generators and buyers respond to changing supply and demand conditions driven by temperature, wind, precipitation, and so on. It will not reveal how weather affects what consumers pay at the meter, because the regulated retail tariff insulates Bulgarian households from short-term wholesale fluctuations. The wholesale market is the appropriate place to study weather-price relationships precisely because it is the part of the system that responds to weather; the retail tariff, by design, smooths these signals out.

- **Provider:** Ember
- **License:** CC-BY-4.0 (free reuse with attribution)
- **URL:** [ember-energy.org/data/european-wholesale-electricity-price-data](https://ember-energy.org/data/european-wholesale-electricity-price-data)
- **Original source:** ENTSO-E Transparency Platform (Bulgarian market: IBEX)
- **Variable used:** Daily mean day-ahead clearing price, nominal €/MWh
- **Coverage:** 2015-01-01 through 2025-12-31

### 2.2 Source 2 — Open-Meteo: Historical Weather

Open-Meteo provides a free historical weather API that serves reanalysis data from the ERA5 dataset, produced by the European Centre for Medium-Range Weather Forecasts (ECMWF) under the Copernicus Climate Change Service. Reanalysis data is not raw observation — it is the output of a physics-based atmospheric model that ingests observations from weather stations, satellites, and balloons, and produces a globally consistent gridded estimate of past weather. This is the standard approach for historical climate research and is preferable to relying on a single weather station, which may have gaps, instrument changes, or local microclimate effects unrepresentative of the broader region.

Five variables are pulled at hourly resolution and aggregated to daily values: temperature (mean), precipitation (sum), wind speed (mean), cloud cover (mean), and snowfall (sum). The aggregation method is chosen to match the physical meaning of each variable — precipitation and snowfall accumulate, while temperature, wind, and cloud cover are averaged.

#### Why five cities, not one

Bulgarian electricity demand is not uniform across the country. A national-scale weather variable is needed to correlate with a national-scale price. Using only Sofia would bias the analysis toward conditions in the capital and miss the contributions of coastal cities (where summer cooling demand peaks) and the Danube plain (which has its own climate regime). Weather is therefore pulled for five Bulgarian cities — Sofia, Plovdiv, Varna, Burgas, and Ruse — chosen to span the country's main population centres and climate zones. The cities are then combined into a single national weather index. The construction of that index, including the weighting scheme and its limitations, is documented in Section 3.

- **Provider:** Open-Meteo (Historical Weather API)
- **License:** CC-BY-4.0
- **URL:** [open-meteo.com/en/docs/historical-weather-api](https://open-meteo.com/en/docs/historical-weather-api)
- **Underlying dataset:** ERA5 reanalysis (ECMWF / Copernicus Climate Change Service)
- **Variables used:** Temperature, precipitation, wind speed, cloud cover, snowfall
- **Resolution:** Hourly, aggregated to daily
- **Cities:** Sofia, Plovdiv, Varna, Burgas, Ruse
- **Coverage:** 2015-01-01 through 2025-12-31

### 2.3 Source 3 — Eurostat: HICP Inflation Index

The Harmonised Index of Consumer Prices (HICP) is the official inflation measure published by Eurostat for all EU member states. National HICP data are collected by each country's National Statistical Institute — for Bulgaria, this is the National Statistical Institute (NSI Bulgaria) — and harmonised by Eurostat under a common methodology that makes the indices comparable across countries. The harmonised methodology is the appropriate index basis here, because Bulgarian wholesale electricity is sold on euro-denominated markets, and the HICP reflects the same currency basis as the price data.

The role of this dataset in the analysis is structural, not exploratory. Nominal prices in 2025 cannot be compared directly to nominal prices in 2015, because a euro in 2025 buys substantially less than a euro in 2015. Failing to deflate the price series would mean that any apparent long-term price trend would conflate two distinct phenomena: real changes in the cost of electricity (the thing we want to study) and the general erosion of the euro's purchasing power (which has nothing to do with weather). Section 3 documents the deflation procedure and shows the difference it makes to the price series.

The HICP is published monthly. Daily prices are deflated using the HICP value for the month in which each price observation falls — this is the standard approach and is appropriate given that the index does not change at daily resolution.

- **Provider:** Eurostat (data collected by NSI Bulgaria)
- **License:** Free reuse with attribution
- **URL:** [ec.europa.eu/eurostat/databrowser/view/PRC_HICP_MIDX](https://ec.europa.eu/eurostat/databrowser/view/PRC_HICP_MIDX)
- **Variable used:** Monthly HICP index for Bulgaria, base year 2015 = 100
- **Coverage:** 2015-01 through 2025-12

### 2.4 Source 4 — Eurostat Urban Audit: City Population

The Urban Audit dataset (`urb_cpop1`) provides annual population statistics for European cities, collected by national statistical institutes — again, NSI Bulgaria for the Bulgarian cities — and published through Eurostat. The dataset covers all five cities used for the weather index.

The role of this dataset is supporting, not substantive. It is used to assign weights when combining the five city-level weather series into a single national weather index. Population is used as a proxy for electricity demand: cities with more people contribute more to national demand and therefore more to the index. This is a defensible-not-correct choice, and its limitations are discussed in Section 3.

A single reference year (2020) is used to compute the weights, which are then held fixed across the entire 2015–2025 analysis period. Bulgarian city populations did shift modestly over the decade — Sofia gained residents while smaller cities lost them but these shifts are small relative to the proxy nature of population-as-demand-weight. Allowing the weights to vary year by year would couple the weather index to demographic drift in a way that complicates the interpretation of the index across time. Holding the weights fixed prioritises a stable, interpretable index over marginal demographic precision. This decision is documented explicitly in Section 3.

- **Provider:** Eurostat Urban Audit (data collected by NSI Bulgaria)
- **License:** Free reuse with attribution
- **URL:** [ec.europa.eu/eurostat/databrowser/view/URB_CPOP1](https://ec.europa.eu/eurostat/databrowser/view/URB_CPOP1)
- **Variable used:** Annual city population, reference year 2020
- **Cities:** Sofia, Plovdiv, Varna, Burgas, Ruse

### 2.5 Why these sources are independent and appropriate

The project requires at least two independent data sources; this analysis uses four. Each is produced by a different organisation or pipeline, using different methods, on different physical or socio-economic phenomena. Ember curates market data from ENTSO-E. Open-Meteo serves ERA5 reanalysis from ECMWF. Eurostat publishes HICP statistics compiled by national statistical institutes. The Urban Audit population data is also Eurostat-published but draws on a structurally separate collection process — population censuses and annual demographic registers, not consumer price surveys.

Independence matters here because the analysis hinges on relationships *between* sources. If the weather data and the price data came from a common underlying model, any correlation found between them could be an artifact of the shared source rather than a genuine market signal. The two sources whose relationship is the central object of study Ember (prices) and Open-Meteo (weather) are produced by completely separate organisations using completely separate methods, and there is no shared upstream pipeline between them.

All four datasets are licensed for free reuse with attribution, which is documented in the References section. The full citation chain — Ember → ENTSO-E → IBEX, Open-Meteo → ERA5 (ECMWF / Copernicus), Eurostat HICP → NSI Bulgaria, Eurostat Urban Audit → NSI Bulgaria — is preserved so that any reader can trace back to the original observations.